In [ ]:
import json

# Create this file with: python scrape.py --pages 10
with open("articles.json", encoding="utf-8") as file:
    articles = json.load(file)

# Combine similar labels for the classifier without changing articles.json.
combined_category = "Campus & University Affairs"
articles = [
    {
        **article,
        "category": combined_category
        if article["category"] in {"Campus Life", "University Affairs"}
        else article["category"],
    }
    for article in articles
]

display(len(articles))

In [ ]:
# Extract all unique, non-null categories
unique_categories = {article["category"] for article in articles if article.get("category")}

# Print them sorted alphabetically
for category in sorted(unique_categories):
    print(category)

In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split
# Splits the articles into 80% train and 20% test
train, test = train_test_split(articles, test_size=0.2, random_state=42)

print(f"Train size: {len(train)}")
print(f"Test size: {len(test)}")

train_counts = Counter(article["category"] for article in train)
test_counts = Counter(article["category"] for article in test)

print("\nCategory counts:")
for category in sorted({article["category"] for article in articles}):
    print(f"{category}: train={train_counts[category]}, test={test_counts[category]}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Extract the text strings and labels from list of dictionaries
train_texts = [article["summary"] for article in train]
test_texts = [article["summary"] for article in test]
train_labels = [article["category"] for article in train]
test_labels = [article["category"] for article in test]

# 2. Vectorize the text
count_vect = CountVectorizer(stop_words="english", max_df=0.7)
X_train_counts = count_vect.fit_transform(train_texts)  # .fit_transform does fit and transform in one step
X_test_counts = count_vect.transform(test_texts)

# 3. View some vocabulary words
feature_names = count_vect.get_feature_names_out()
print("Sample vocabulary:", feature_names[:10])

# 4. See the shape of the training data matrix (Rows = articles, Columns = unique words)
print("X_train_counts shape:", X_train_counts.shape)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

classifier = MultinomialNB()

# Computing probabilities
classifier.fit(X_train_counts, train_labels)

# Use Bayes Theorem to predict
y_pred = classifier.predict(X_test_counts)

# for pred, act, text in zip(y_pred, test_labels, test_texts):
#     print(f"Predicted: {pred}, Actual: {act}")

p,r,f,s = precision_recall_fscore_support(test_labels, y_pred, average='weighted')
print(f"precision={p},recall={r},fscore={f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

labels = sorted(unique_categories)
matrix = confusion_matrix(test_labels, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(matrix, display_labels=labels).plot(
    ax=ax, cmap="Blues", xticks_rotation=45
)
ax.set_title("Multinomial Naive Bayes Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import math
import matplotlib.pyplot as plt

top_n = 10
categories = sorted(unique_categories)
columns = 2
rows = math.ceil(len(categories) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(14, 4 * rows))

for axis, category in zip(axes.flat, categories):
    category_texts = [
        article["summary"] for article in articles if article["category"] == category
    ]
    counts = count_vect.transform(category_texts).sum(axis=0).A1
    words = count_vect.get_feature_names_out()
    top_indices = counts.argsort()[-top_n:][::-1]

    axis.barh(words[top_indices][::-1], counts[top_indices][::-1])
    axis.set_title(category)
    axis.set_xlabel("Count")

for axis in axes.flat[len(categories):]:
    axis.set_visible(False)

fig.suptitle("Most Common Words by Category", fontsize=16)
plt.tight_layout()
plt.show()